# 02 · Trayectorias de entrenamiento

**Qué contiene.** Arma las conversaciones de entrenamiento (pregunta → `search_series` → `get_series` → `submit_answer`). Claude Haiku 5.5 escribe solo la consulta de búsqueda; todo lo demás sale del catálogo y de la API. Celda 0: construcción; celda 1: chequeo del formato con el tokenizer de Qwen.

| | |
|---|---|
| **Runtime** | CPU (sin GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `catalogo_busqueda.csv`, `specs_train_v1.csv`, `train_final2.csv` |
| **Salidas** | `consultas_busqueda_train_parcial.csv`, `trayectorias_train_v1.jsonl`, `tools_v1.json` |
| **Secretos de Colab** | `ANTHROPIC_API_KEY` |
| **Costo y tiempo** | USD 0,09 y unos 400 s en la corrida publicada (1.853 de 1.863 consultas generadas) |
| **Orden** | Después de `01`; antes de `06` |
| **En el repositorio** | `esquema.py`, `trayectorias.py`, `scripts/construir_trayectorias.py` (con `--solo-armar` reproduce el conjunto sin llamar a la API) |

**Advertencias.**
- Con `N_PRUEBA = None` (como está) procesa todas las preguntas y llama a la API de pago. Para probar, poner un número chico.
- Reproducir este conjunto sin gastar: usar el script con `--solo-armar`, que toma las consultas ya guardadas.


In [ ]:
!pip -q install -U anthropic
import os, re, json, time, threading, shutil, sqlite3, unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import anthropic
from google.colab import drive, userdata

print("CELDA: TRAYECTORIAS DE ENTRENAMIENTO (TRAIN)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
N_PRUEBA = None         # primera corrida de revisión; después ponerlo en None
PCT_SIN_DATOS = 0.18
HILOS = 4
MODEL = "claude-haiku-5-5"
PARCIAL = f"{RUTA}/consultas_busqueda_train_parcial.csv"
SALIDA = f"{RUTA}/trayectorias_train_v1.jsonl"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
ABORT = threading.Event()

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

# ---------- Herramientas (formato OpenAI) ----------
TOOLS = [
    {"type": "function", "function": {
        "name": "search_series",
        "description": "Busca series estadísticas en el catálogo por palabras clave. Devuelve hasta 10 candidatas.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Palabras clave del tema, sin fechas."}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_series",
        "description": "Consulta datos de una serie de la API de Series de Tiempo. Fechas como 'AAAA', 'AAAA-MM' o 'AAAA-MM-DD'.",
        "parameters": {"type": "object", "properties": {
            "ids": {"type": "string"},
            "start_date": {"type": "string"},
            "end_date": {"type": "string"},
            "representation_mode": {"type": "string",
                                    "enum": ["value", "change", "percent_change", "percent_change_a_year_ago"]},
            "limit": {"type": "integer"},
            "sort": {"type": "string", "enum": ["asc", "desc"]}},
            "required": ["ids"]}}},
    {"type": "function", "function": {
        "name": "submit_answer",
        "description": "Entrega la respuesta final. 'valor' y 'fecha' van en null si la API no devolvió datos.",
        "parameters": {"type": "object", "properties": {
            "respuesta": {"type": "string"},
            "valor": {"type": ["number", "null"]},
            "fecha": {"type": ["string", "null"]}},
            "required": ["respuesta"]}}},
]
with open(f"{RUTA}/tools_v1.json", "w") as f:
    json.dump(TOOLS, f, ensure_ascii=False, indent=1)

# ---------- Preguntas a usar ----------
E = pd.read_csv(f"{RUTA}/specs_train_v1.csv")
E["spec_id"] = E.index
F = pd.read_csv(f"{RUTA}/train_final2.csv")
F = F[F["marca_var"].fillna("") != "var_sobre_tasa"]
F = F.merge(E[["spec_id", "data", "respuesta_final", "serie_titulo", "serie_unidades"]],
            on="spec_id", how="left")
rows = []
for r in F.to_dict("records"):
    rows.append({**r, "estilo": "tecnica", "pregunta": r["tecnica"]})
    if not r["coloquial_ambigua"]:
        rows.append({**r, "estilo": "coloquial", "pregunta": r["coloquial"]})
Q = pd.DataFrame(rows)
non = Q[Q["tipo"] != "sin_datos"]
sin = Q[(Q["tipo"] == "sin_datos") & (Q["estilo"] == "tecnica")]   # sin_datos solo con preguntas técnicas
n_sin = int(round(PCT_SIN_DATOS / (1 - PCT_SIN_DATOS) * len(non)))
sin = sin.sample(n=min(n_sin, len(sin)), random_state=3)
Q = pd.concat([non, sin]).sample(frac=1, random_state=5).reset_index(drop=True)
Q["qid"] = Q["spec_id"].astype(str) + "-" + Q["estilo"]
print(f"Preguntas candidatas: {len(Q)} | por tipo: {Q['tipo'].value_counts().to_dict()}")
if N_PRUEBA:
    Q = Q.head(N_PRUEBA)
    print("MODO PRUEBA:", len(Q), "preguntas")

cat = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
info = cat.drop_duplicates("serie_id").set_index("serie_id")

# ---------- Consulta de búsqueda (Haiku, sin razonamiento) ----------
def g(c, col):
    v = c[col]
    return "" if pd.isna(v) else str(v)

def prompt_consulta(r):
    c = info.loc[r["serie_id"]]
    return f"""Un usuario le hizo esta pregunta a un asistente que consulta un catálogo de series estadísticas argentinas.
Escribí la consulta de búsqueda (entre 2 y 7 palabras, sin fechas ni números) que conviene teclear en el buscador del catálogo, que busca por palabras en títulos y descripciones.
Usá las palabras clave del tema de la pregunta. Podés agregar un término técnico equivalente si es habitual en estadísticas oficiales, pero no copies el título técnico.

Pregunta: {r['pregunta']}

Dato al que se refiere (solo para orientarte):
Título: {g(c, 'serie_titulo')} | Descripción: {g(c, 'serie_descripcion')} | Dataset: {g(c, 'dataset_titulo')}

Devolvé solo JSON: {{"consulta": "..."}}"""

def tarea(r):
    if ABORT.is_set():
        return {"qid": r["qid"], "ok": False, "error": "abortado"}
    try:
        for _ in range(3):
            msg = client.messages.create(model=MODEL, max_tokens=200,
                                         messages=[{"role": "user", "content": prompt_consulta(r)}],
                                         extra_body={"thinking": {"type": "disabled"}})
            txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
            try:
                q = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])["consulta"].strip()
            except Exception:
                continue
            if 1 <= len(q.split()) <= 8 and not re.search(r"\d", q):
                return {"qid": r["qid"], "ok": True, "error": "", "consulta": q,
                        "ent": msg.usage.input_tokens, "sal": msg.usage.output_tokens}
        return {"qid": r["qid"], "ok": False, "error": "sin consulta válida"}
    except anthropic.AuthenticationError as e:
        ABORT.set()
        return {"qid": r["qid"], "ok": False, "error": f"AuthenticationError (se corta todo): {str(e)[:80]}"}
    except Exception as e:
        return {"qid": r["qid"], "ok": False, "error": f"{type(e).__name__}: {str(e)[:100]}"}

prev = pd.read_csv(PARCIAL) if os.path.exists(PARCIAL) else pd.DataFrame(columns=["qid", "ok", "consulta"])
if len(prev):
    prev = prev[prev["ok"] == True]
hechos = set(prev["qid"])
pend = [r for r in Q.to_dict("records") if r["qid"] not in hechos]
print(f"Consultas ya hechas: {len(hechos)} | pendientes: {len(pend)}")

t0, res = time.time(), []
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    for k, out in enumerate(ex.map(tarea, pend), 1):
        res.append(out)
        if k % 200 == 0:
            pd.concat([prev, pd.DataFrame(res)], ignore_index=True).to_csv(PARCIAL, index=False)
            print(f"  [{k}/{len(pend)}] {time.time() - t0:.0f}s", flush=True)
R = pd.DataFrame(res)
todo = pd.concat([prev, R], ignore_index=True) if len(R) else prev
todo.to_csv(PARCIAL, index=False)
if len(R):
    costo = R["ent"].fillna(0).sum() * 0.10 / 1e6 + R["sal"].fillna(0).sum() * 0.50 / 1e6
    print(f"Esta corrida: {int(R['ok'].sum())}/{len(R)} ok | costo ${costo:.4f} | {time.time() - t0:.0f}s")
    print("Fallos:", Counter(R[~R['ok']]['error']).most_common(3))
assert not ABORT.is_set(), "Se cortó por error de autenticación: revisá la clave y volvé a ejecutar"

# ---------- Buscador real (índice de Drive, variante de prefijos) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)
STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows_ = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                        "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [x[0] for x in rows_]

# ---------- Armado de trayectorias ----------
def periodo(fecha, freq):
    return fecha[:4] if freq == "anual" else fecha[:7]

def args_submit(r, d, params):
    if r["tipo"] == "sin_datos":
        txt = re.sub(r"\s*La serie tiene datos entre.*$", "", r["respuesta_final"])
        return {"respuesta": txt, "valor": None, "fecha": None}
    if r["tipo"] == "ultimo_dato":
        return {"respuesta": r["respuesta_final"], "valor": d[0][1], "fecha": d[0][0]}
    if r["tipo"] == "valor_puntual":
        return {"respuesta": r["respuesta_final"], "valor": d[0][1], "fecha": params["start_date"]}
    return {"respuesta": r["respuesta_final"], "valor": round(d[-1][1] * 100, 2),
            "fecha": periodo(d[-1][0], r["frecuencia"])}

def llamada(i, nombre, args):
    return {"role": "assistant", "content": "",
            "tool_calls": [{"id": f"call_{i}", "type": "function",
                            "function": {"name": nombre, "arguments": args}}]}

buenas = todo[todo["ok"] == True]
consultas = dict(zip(buenas["qid"], buenas["consulta"]))
trayectorias, perdidas = [], Counter()
for r in Q.to_dict("records"):
    consulta = consultas.get(r["qid"])
    if not consulta:
        perdidas["sin consulta generada"] += 1
        continue
    ids = buscar(consulta, 10)
    if r["serie_id"] not in ids:
        perdidas[f"la serie no aparece en el top 10 ({r['estilo']})"] += 1
        continue
    d = json.loads(r["data"])
    params = json.loads(r["params"])
    cand = [{"serie_id": s, "titulo": str(info.loc[s, "serie_titulo"]),
             "distribucion": str(info.loc[s, "distribucion_titulo"]),
             "frecuencia": str(info.loc[s, "frecuencia"])} for s in ids]
    get_res = {"status": 200, "serie_id": r["serie_id"], "titulo": r["serie_titulo"],
               "unidades": None if pd.isna(r["serie_unidades"]) else r["serie_unidades"],
               "frecuencia": r["frecuencia"], "data": d}
    msgs = [{"role": "user", "content": r["pregunta"]},
            llamada(1, "search_series", {"query": consulta}),
            {"role": "tool", "tool_call_id": "call_1", "name": "search_series",
             "content": json.dumps(cand, ensure_ascii=False)},
            llamada(2, "get_series", params),
            {"role": "tool", "tool_call_id": "call_2", "name": "get_series",
             "content": json.dumps(get_res, ensure_ascii=False)},
            llamada(3, "submit_answer", args_submit(r, d, params))]
    trayectorias.append({"qid": r["qid"], "spec_id": int(r["spec_id"]), "tipo": r["tipo"],
                         "estilo": r["estilo"], "serie_id": r["serie_id"], "familia": r["familia"],
                         "rango_busqueda": ids.index(r["serie_id"]) + 1, "messages": msgs})

print(f"\nTrayectorias: {len(trayectorias)} de {len(Q)} | perdidas: {dict(perdidas)}")
T = pd.DataFrame([{k: v for k, v in t.items() if k != "messages"} for t in trayectorias])
if len(T):
    print("Por tipo:", T["tipo"].value_counts().to_dict())
    print("Por estilo:", T["estilo"].value_counts().to_dict())
    print(f"% sin_datos: {(T['tipo'] == 'sin_datos').mean():.1%} | familias: {T['familia'].nunique()} | series: {T['serie_id'].nunique()}")
    print("Rango del buscador (mediana, máx):", int(T["rango_busqueda"].median()), int(T["rango_busqueda"].max()))
    print("\nEjemplos de consultas generadas:")
    for t in trayectorias[:5]:
        print(f"  [{t['estilo']}|{t['tipo']}] {t['messages'][0]['content'][:90]}  ->  {t['messages'][1]['tool_calls'][0]['function']['arguments']['query']}")
    for tipo in ["variacion_interanual", "sin_datos"]:
        ej = next((t for t in trayectorias if t["tipo"] == tipo), None)
        if ej:
            print(f"\n--- Trayectoria completa de ejemplo ({tipo}):")
            for m in ej["messages"]:
                if m["role"] == "tool":
                    print("  TOOL:", m["content"][:260])
                elif m.get("tool_calls"):
                    fn = m["tool_calls"][0]["function"]
                    print("  LLAMADA:", fn["name"], json.dumps(fn["arguments"], ensure_ascii=False)[:260])
                else:
                    print("  USER:", m["content"])
    with open(SALIDA, "w") as f:
        for t in trayectorias:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print("\nGuardado:", SALIDA, "y tools_v1.json")

In [ ]:
!pip -q install transformers
import json
import numpy as np
from google.colab import drive
from transformers import AutoTokenizer

print("CELDA: CHEQUEO DE FORMATO CON EL TOKENIZER DE QWEN (CPU, SIN COSTO)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
MODELO = "Qwen/Qwen3-4B-Instruct-2507"

tok = AutoTokenizer.from_pretrained(MODELO)
tools = json.load(open(f"{RUTA}/tools_v1.json"))
T = [json.loads(l) for l in open(f"{RUTA}/trayectorias_train_v1.jsonl")]
print("Trayectorias:", len(T))

# 1) Una trayectoria renderizada como texto
ej = next(t for t in T if t["tipo"] == "variacion_interanual")
texto = tok.apply_chat_template(ej["messages"], tools=tools, tokenize=False)
print("\n" + "=" * 70 + "\n" + texto[:6000] + "\n" + "=" * 70)

# 2) Comprobaciones de estructura sobre el ejemplo
print("\nLlamadas a herramientas (<tool_call>):", texto.count("<tool_call>"), "(esperado 3)")
print("Respuestas de herramientas (<tool_response>):", texto.count("<tool_response>"), "(esperado 2)")
print("Argumentos como JSON:", '"name": "get_series"' in texto)

# 3) Render y largo de TODAS las trayectorias
largos, errores = [], 0
for t in T:
    try:
        s = tok.apply_chat_template(t["messages"], tools=tools, tokenize=False)
        largos.append(len(tok(s, add_special_tokens=False)["input_ids"]))
    except Exception:
        errores += 1
L = np.array(largos)
print(f"\nSe renderizaron {len(L)} de {len(T)} (errores: {errores})")
print("Tokens por trayectoria: media %.0f | mediana %d | p95 %d | p99 %d | máx %d" %
      (L.mean(), np.median(L), np.percentile(L, 95), np.percentile(L, 99), L.max()))

# 4) Costo fijo: lo que ocupan solo las herramientas y el mensaje de sistema
base = tok.apply_chat_template([{"role": "user", "content": "hola"}], tools=tools, tokenize=False)
print("Tokens de la descripción de herramientas + sistema:", len(tok(base, add_special_tokens=False)["input_ids"]))